# RQ1: validation improvement of UMDA4GI against First-Improvement, Random Search and GP

**Definitions** (the same in every notebook, see the docstring of `analysis.py`):

- *improvement*: % change in CPU cycles of the returned patch on the validation inputs (negative = faster).
  A patch that fails the correctness oracle on the validation inputs counts as 0% (the original program is kept).
- *gap*: for each run, validation change minus training change, in percentage points (positive = the patch
  generalises worse than it trained). Aggregated with the median of the runs.
- *cell*: (software, algorithm, budget), 15 runs = 5 folds × 3 seeds. Each run has 1000 evaluations; the budgets
  100, 250, 500 and 750 are checkpoints of the same run.
- *ranking*: per software, the median of each cell, averaged over the five budgets and ranked; rank sum over the
  four systems (lower = better).

In [1]:
import pathlib
import sys

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(pathlib.Path('..', 'revision_analysis').resolve()))
import analysis as A

RESULTS = 'results.csv'           # one row per run and budget (checkpoint)
OUT = pathlib.Path('analysis')    # tables written by analysis.py
REFRESH = False                   # True: recompute every table from results.csv
if REFRESH or not (OUT / 'cells.csv').exists():
    A.run(RESULTS, OUT, 'zero')
df = A.load(RESULTS, 'zero')
cells = pd.read_csv(OUT / 'cells.csv')
SW = {'minisat_hack': 'MiniSAT-hack', 'sat4j': 'Sat4J', 'weka': 'Weka', 'pngOptim': 'OptiPNG'}
pd.set_option('display.precision', 2)
pd.set_option('display.max_rows', 300)
pd.set_option('display.width', 200)

## Median improvement per software, algorithm and budget

Median of the 15 runs with its bootstrap 95% confidence interval (2000 resamples).

In [2]:
c = cells[cells.group.isin(A.BASE)].copy()
c['value'] = c.apply(lambda r: f"{r.improvement_median:.2f} [{r.improvement_ci_low:.2f}, {r.improvement_ci_high:.2f}]", axis=1)
display(c.pivot_table(index=['software', 'group'], columns='budget', values='value', aggfunc='first'))

budget                              100                    250                    500                     750                    1000
software     group                                                                                                                   
minisat_hack FI     -0.62 [-19.08, 1.56]  -10.84 [-17.62, 0.99]  -9.60 [-38.16, -0.82]  -29.78 [-38.57, 14.08]  -29.78 [-37.79, 5.50]
             GP       3.23 [-1.64, 9.09]    2.85 [-1.89, 12.16]   0.70 [-21.38, 11.79]   -1.70 [-16.64, 15.91]  -3.98 [-32.04, 13.75]
             RS      1.46 [-2.32, 18.59]    2.23 [-8.42, 10.33]    7.42 [-2.21, 21.25]    0.00 [-35.41, 15.19]   0.00 [-35.41, 23.01]
             UMDA    -0.56 [-3.21, 2.99]     0.61 [-4.00, 2.99]     0.35 [-4.00, 5.44]     -0.68 [-4.00, 5.61]  -0.39 [-20.14, 11.74]
pngOptim     FI       0.02 [-0.08, 0.09]    -0.01 [-0.04, 0.06]    -0.04 [-0.19, 0.06]     -0.04 [-0.12, 0.04]    -0.04 [-0.18, 0.03]
             GP       0.04 [-0.00, 0.07]     0.04 [-0.00, 0.07]     0.05 [-0.00, 0.10]     -0.03 [-0.09, 0.02]    -0.01 [-0.08, 0.02]
             RS       0.02 [-0.02, 0.17]     0.07 [-0.01, 0.26]     0.06 [-0.09, 0.14]      0.03 [-0.07, 0.12]     0.01 [-0.14, 0.10]
             UMDA     0.00 [-0.12, 0.09]    -0.02 [-0.14, 0.09]    -0.02 [-0.12, 0.02]     -0.03 [-0.14, 0.08]    -0.12 [-0.14, 0.00]
sat4j        FI       0.36 [-1.65, 2.08]    -0.36 [-1.93, 2.11]     0.86 [-1.93, 2.08]      0.86 [-2.02, 2.11]    -1.17 [-2.09, 2.08]
             GP      -0.58 [-2.93, 0.98]     0.93 [-1.63, 2.30]    -0.88 [-2.92, 2.30]     -1.63 [-4.16, 1.40]    -1.63 [-2.90, 2.31]
             RS      -1.73 [-4.60, 0.42]    -1.73 [-5.54, 0.29]     0.29 [-2.85, 3.72]      0.29 [-1.73, 3.72]     0.29 [-2.50, 4.39]
             UMDA    -0.70 [-1.04, 0.64]    -1.16 [-2.88, 0.33]    -0.77 [-2.88, 0.33]     -0.22 [-2.88, 0.93]     0.04 [-2.88, 1.60]
weka         FI       0.00 [-1.07, 1.36]     1.23 [-1.07, 2.24]    -0.56 [-1.34, 0.98]     -0.52 [-1.34, 0.67]    -0.81 [-1.38, 0.73]
             GP      -1.07 [-2.15, 0.47]     0.01 [-1.64, 0.95]     0.14 [-0.48, 0.86]      0.14 [-1.07, 0.95]     0.14 [-1.80, 1.05]
             RS      -1.67 [-3.26, 0.65]   -0.86 [-3.18, -0.07]   -2.07 [-3.26, -0.14]    -2.07 [-3.26, -0.69]   -2.07 [-3.18, -0.69]
             UMDA   -0.64 [-1.16, -0.08]   -0.50 [-1.16, -0.08]   -0.64 [-1.56, -0.30]    -0.64 [-1.16, -0.30]    -0.66 [-3.07, 0.01]

## Friedman test per software and budget

Over the four algorithms, with the 15 (fold, seed) pairs as blocks.

In [3]:
fr = pd.read_csv(OUT / 'rq1_rq2_friedman.csv')
fr = fr[fr.metric == 'improvement']
display(fr.pivot(index='software', columns='budget', values='p'))
print('significant (p < 0.05):', (fr.p < 0.05).sum(), 'of', len(fr))

budget,100,250,500,750,1000
software,,,,,
minisat_hack,0.88,0.33,0.06,0.65,0.98
pngOptim,0.88,0.16,0.06,0.26,0.31
sat4j,0.46,0.23,0.88,0.39,0.72
weka,0.78,0.09,0.07,0.14,0.61


significant (p < 0.05): 0 of 20


## Pairwise comparisons

Wilcoxon signed-rank tests paired by fold and seed, Holm's correction within each (software, budget, metric), i.e. over its 6 pairs. `a12_a_better` is the probability that a run of `group_a` gives a lower (better) value than a run of `group_b`.

In [4]:
pw = pd.read_csv(OUT / 'rq1_rq2_pairwise.csv')
p = pw[pw.metric == 'improvement']
display(p[p.p_holm < 0.05])
print('significant after Holm within each family:', (p.p_holm < 0.05).sum(), 'of', len(p))

,software,budget,group_a,group_b,metric,blocks,median_a,median_b,a12_a_better,p,p_holm


significant after Holm within each family: 0 of 120


## Ranking

In [5]:
display(pd.read_csv(OUT / 'rq1_improvement_ranking.csv'))

,group,minisat_hack_value,pngOptim_value,sat4j_value,weka_value,minisat_hack_rank,pngOptim_rank,sat4j_rank,weka_rank,rank_sum
0,UMDA,-0.13,-0.04,-0.56,-0.62,2.0,1.0,2.0,2.0,7.0
1,FI,-16.12,-0.02,0.11,-0.13,1.0,2.0,4.0,3.0,10.0
2,GP,0.22,0.02,-0.76,-0.13,3.0,3.0,1.0,4.0,11.0
3,RS,2.22,0.04,-0.52,-1.75,4.0,4.0,3.0,1.0,12.0


### Sensitivity: the same ranking with the mean instead of the median

The median is used because a few extreme runs would dominate the mean (see below); the tests are rank-based and do not depend on this choice.

In [6]:
means = (df[df.group.isin(A.BASE)].groupby(['software', 'group', 'checkpoint'])['improvement'].mean()
         .rename('improvement_mean').reset_index())
display(A.ranking(means, A.BASE, 'improvement_mean'))

software,group,minisat_hack_value,pngOptim_value,sat4j_value,weka_value,minisat_hack_rank,pngOptim_rank,sat4j_rank,weka_rank,rank_sum
0,UMDA,-1.42,-1.24,0.49,-0.82,1.0,2.0,2.0,2.0,7.0
1,FI,8.08,-2.02,0.17,-0.21,4.0,1.0,1.0,4.0,10.0
2,RS,5.16,-0.68,1.67,-1.70,3.0,3.0,3.0,1.0,10.0
3,GP,-0.84,0.02,2.68,-0.37,2.0,4.0,4.0,3.0,13.0


## Why the median: extreme runs

On MiniSAT, one validation fold gives improvements close to −90%, while some patches make the solver several times slower on other folds.

In [7]:
m = df[df.group.isin(A.BASE) & (df.software == 'minisat_hack')]['improvement']
print(f'MiniSAT: min {m.min():.1f}%, max {m.max():.1f}%, runs x budgets above +50%: {(m > 50).sum()} of {len(m)}')

MiniSAT: min -90.0%, max 1005.1%, runs x budgets above +50%: 35 of 300


## Pairwise tests involving UMDA4GI before any correction, and with Holm over all 240

In [8]:
pw = pd.read_csv(OUT / 'rq1_rq2_pairwise.csv')
u = pw[((pw.group_a == 'UMDA') | (pw.group_b == 'UMDA')) & (pw.p < 0.05)]
favours = ((u.group_a == 'UMDA') & (u.median_a < u.median_b)) | ((u.group_b == 'UMDA') & (u.median_b < u.median_a))
print(f'comparisons with UMDA4GI and p < 0.05: {len(u)}, in favour of UMDA4GI (lower median): {favours.sum()}')
display(u[~favours])
print('smallest p after Holm over all', len(pw), 'comparisons:', A.holm(pw.p).min().round(3))

comparisons with UMDA4GI and p < 0.05: 20, in favour of UMDA4GI (lower median): 18


,software,budget,group_a,group_b,metric,blocks,median_a,median_b,a12_a_better,p,p_holm
101,pngOptim,750,GP,UMDA,improvement,15,-0.03,-0.03,0.41,0.05,0.29
130,sat4j,100,RS,UMDA,gap,15,3.02,5.64,0.63,0.03,0.15


smallest p after Holm over all 240 comparisons: 0.146


## Runs without an improving patch

A run that finds no improving patch returns the original program and counts as 0%.

In [9]:
last = df[df.checkpoint == 1000]
print('runs with no improving patch at 1000 evaluations:', (last.validate_status == 'NO_PATCH').sum(), 'of', len(last))
display(last[last.validate_status == 'NO_PATCH'][['software', 'group', 'fold', 'seed']])

runs with no improving patch at 1000 evaluations: 5 of 780


,software,group,fold,seed
1879,sat4j,ABL-GP-2,1,1
2209,weka,UMDA,2,102
3219,pngOptim,UMDA,4,204
3344,pngOptim,UMDA-llama8b,4,104
3669,pngOptim,ABL-UMDA-k1,4,204


## OptiPNG: the one edit that improves it

The median validation improvement on OptiPNG is within 0.12 percentage points of zero for every algorithm, except in the runs whose patch deletes statement 424 of `optim.c`, the call `png_set_compression_window_bits(..., 9)` that sets a 512-byte compression window for two of OptiPNG's strategies.

In [10]:
c = cells[(cells.software == 'pngOptim') & cells.group.isin(A.BASE)]
print('largest |median validation improvement| on OptiPNG:', c.improvement_median.abs().max().round(3))
w = df[df.best_patch.fillna('').str.contains("XmlNodeDeletion<stmt>(('optim.c.xml', 'stmt', 424))", regex=False)]
print('runs whose patch deletes it:', w.id.nunique())
display(w[w.checkpoint == 1000][['group', 'fold', 'seed', 'change_perc_train', 'improvement']])

largest |median validation improvement| on OptiPNG: 0.117
runs whose patch deletes it: 18


,group,fold,seed,change_perc_train,improvement
2929,FI,1,1,-16.92,-23.36
2939,FI,3,3,-17.08,-26.60
3014,RS,3,3,-17.00,-27.95
3044,RS,4,104,-18.58,-14.81
3164,UMDA,3,3,-17.17,-28.06
3259,UMDA-qwen7b,2,102,-15.51,-22.57
3269,UMDA-qwen7b,4,104,-18.68,-14.16
3349,UMDA-llama8b,5,105,-18.68,-15.09
3419,LLM-qwen7b,4,104,-18.44,-14.83
3464,LLM-llama8b,3,3,-17.09,-28.13


## Search time

Minutes until the k-th evaluation of each run (median of the 15 runs), averaged over the five budgets. The runs shared the cluster nodes, so the times include that load.

In [11]:
t = cells[cells.group.isin(A.BASE)].groupby(['group', 'software'])['search_minutes_median'].mean().unstack()
display(t)
display(t.rank(method='min').assign(rank_sum=lambda r: r.sum(axis=1)).sort_values('rank_sum'))

software,minisat_hack,pngOptim,sat4j,weka
group,,,,
FI,780.87,478.45,884.28,901.68
GP,624.45,431.08,739.62,756.87
RS,884.02,448.38,943.53,899.09
UMDA,598.24,545.45,735.19,393.37


software,minisat_hack,pngOptim,sat4j,weka,rank_sum
group,,,,,
GP,2.0,1.0,2.0,2.0,7.0
UMDA,1.0,4.0,1.0,1.0,7.0
FI,3.0,3.0,3.0,4.0,13.0
RS,4.0,2.0,4.0,3.0,13.0
